# **Algorithmic Reasoning Distillation: Hugging Face PEFT QLoRA Pipeline**
### **IT488 Course Project: Midsemester Evaluation (Phases 1 & 2 - 40% Scope)**
**Author:** Pranav Bhat P (Roll No: 231IT049)  
**Course:** IT488  
**Evaluation Target:** Midsemester Evaluation  
**Pipeline:** Pipeline A — Standard Hugging Face PEFT + BitsAndBytes (4-bit NF4 Quantization)

---
### **📌 Pipeline Architecture**
1. **Dataset Engine (Phase 1)**: 100 competitive programming problems from **TACO (`BAAI/TACO`)** (800–1200 rating) across 10 algorithmic categories.
2. **Google Gemini Teacher Supervision**: High-fidelity structured reasoning rationales from `gemini-3.5-flash-lite`.
3. **Rationale-Consistency Filter (Phase 2)**: Semantic paradigm alignment validation.
4. **Stage 1 QLoRA Distillation**: Hugging Face `peft` + `bitsandbytes` 4-bit NF4 fine-tuning on `Qwen/Qwen2.5-0.5B`.
5. **Subprocess Sandbox Judge**: Timeout (2.0s) & memory (512MB) isolated execution judge.
6. **Empirical Evaluation Dashboard**: Top-1 and Top-3 accuracy, Macro F1, VRAM profiling, and Live Interactive Solver.

## **Step 1: Install Dependencies & Setup Environment**
Installs the standard Hugging Face PEFT stack (`transformers`, `peft`, `accelerate`, `bitsandbytes`).

In [1]:
# Install pure Hugging Face stack (no Unsloth dependencies/monkey-patches)
!pip install -q transformers peft accelerate bitsandbytes datasets scikit-learn rich seaborn matplotlib google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 20.6 MB/s eta 0:00:00


In [2]:
import os, sys, time, json, re, math, random, tempfile, subprocess, resource, gc
from dataclasses import dataclass, field, asdict
from enum import Enum
from typing import List, Dict, Any, Optional, Tuple, Union
from collections import defaultdict

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM Available: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    print(f"bfloat16 Supported: {torch.cuda.is_bf16_supported()}")
else:
    print("Running on CPU (Tip: In Colab, select Runtime -> Change runtime type -> T4 GPU)")

def get_gpu_memory_mb():
    if torch.cuda.is_available():
        return torch.cuda.max_memory_allocated() / (1024 * 1024)
    return 0.0

PyTorch Version: 2.11.0+cu128
CUDA Available: True
Active GPU: Tesla T4
VRAM Available: 14.56 GB
bfloat16 Supported: True


## **Step 2: Load 100-Problem TACO Dataset from JSON**
Upload `taco_100_distillation_dataset.json` (or `taco_100_problems.json` + `teacher_traces_100.json`) to Colab's `/content/` directory.
The code loads the 100 TACO benchmark problems and Gemini teacher reasoning traces across 10 categories.

In [4]:
TARGET_TAG_TAXONOMY = [
    "dynamic programming",
    "greedy",
    "graphs",
    "math",
    "data structures",
    "trees",
    "brute force",
    "strings",
    "number theory",
    "binary search",
]
TAG_TO_IDX = {tag: i for i, tag in enumerate(TARGET_TAG_TAXONOMY)}
IDX_TO_TAG = {i: tag for i, tag in enumerate(TARGET_TAG_TAXONOMY)}

@dataclass
class DatasetItem:
    problem_id: str
    title: str
    ground_truth_tag: str
    rating: int
    statement: str
    input_spec: str
    output_spec: str
    teacher_rationale: str
    teacher_solution: str
    teacher_stated_tag: str
    sample_tests: List[Dict[str, str]]
    time_limit_ms: int = 2000
    memory_limit_mb: int = 512
    is_consistency_verified: bool = False

candidate_paths = [
    "taco_100_distillation_dataset.json",
    "data/taco_100_distillation_dataset.json",
    "/content/taco_100_distillation_dataset.json",
]

dataset_file = next((p for p in candidate_paths if os.path.exists(p)), None)
raw_data = None

# Auto-merge fallback if user uploaded taco_100_problems.json and teacher_traces_100.json separately
if dataset_file is None:
    prob_candidates = ["taco_100_problems.json", "data/taco_100_problems.json", "/content/taco_100_problems.json"]
    trace_candidates = ["teacher_traces_100.json", "data/teacher_traces_100.json", "/content/teacher_traces_100.json"]
    p_file = next((p for p in prob_candidates if os.path.exists(p)), None)
    t_file = next((p for p in trace_candidates if os.path.exists(p)), None)
    if p_file and t_file:
        print(f"Detected separate files: {p_file} and {t_file}. Auto-merging...")
        with open(p_file, 'r', encoding='utf-8') as f: p_list = json.load(f)
        with open(t_file, 'r', encoding='utf-8') as f: t_list = json.load(f)
        t_map = {t['problem_id']: t for t in t_list}
        raw_data = []
        for prob in p_list:
            t = t_map.get(prob['problem_id'], {})
            raw_data.append({
                **prob,
                'teacher_rationale': t.get('rationale', ''),
                'teacher_solution': t.get('solution_code', prob.get('raw_solution', '')),
                'teacher_stated_tag': t.get('stated_algorithmic_strategy', prob['ground_truth_tag'])
            })

if dataset_file is None and raw_data is None:
    print("Dataset JSON not found in directory.")
    try:
        from google.colab import files
        print("Please upload 'taco_100_distillation_dataset.json' below:")
        uploaded = files.upload()
        dataset_file = list(uploaded.keys())[0]
    except Exception as e:
        raise FileNotFoundError("Please upload 'taco_100_distillation_dataset.json' to Colab!")

if raw_data is None and dataset_file is not None:
    print(f"Loading dataset from: {dataset_file}")
    with open(dataset_file, "r", encoding="utf-8") as f:
        raw_data = json.load(f)
    if isinstance(raw_data, dict) and 'data' in raw_data:
        raw_data = raw_data['data']
    if isinstance(raw_data, dict) and 'problems' in raw_data:
        raw_data = raw_data['problems']

dataset = []
for d in raw_data:
    dataset.append(DatasetItem(
        problem_id=d["problem_id"],
        title=d.get("title", "Problem"),
        ground_truth_tag=d["ground_truth_tag"],
        rating=d.get("rating", 1000),
        statement=d["statement"],
        input_spec=d.get("input_spec", "Standard competitive programming input format."),
        output_spec=d.get("output_spec", "Standard competitive programming output format."),
        teacher_rationale=d.get("teacher_rationale", f"Step-by-step logic for {d['ground_truth_tag']}."),
        teacher_solution=d.get("teacher_solution", d.get("raw_solution", "")),
        teacher_stated_tag=d.get("teacher_stated_tag", d["ground_truth_tag"]),
        sample_tests=d.get("sample_tests", []),
        time_limit_ms=d.get("time_limit_ms", 2000),
        memory_limit_mb=d.get("memory_limit_mb", 512)
    ))

print(f"✓ Successfully loaded {len(dataset)} problems across 10 algorithmic categories:\n")
for i, tag in enumerate(TARGET_TAG_TAXONOMY, start=1):
    count = sum(1 for item in dataset if item.ground_truth_tag == tag)
    print(f"  {i:2d}. {tag.title():<22} : {count} problems")

Loading dataset from: taco_100_distillation_dataset.json
✓ Successfully loaded 100 problems across 10 algorithmic categories:

   1. Dynamic Programming    : 10 problems
   2. Greedy                 : 10 problems
   3. Graphs                 : 10 problems
   4. Math                   : 10 problems
   5. Data Structures        : 10 problems
   6. Trees                  : 10 problems
   7. Brute Force            : 10 problems
   8. Strings                : 10 problems
   9. Number Theory          : 10 problems
  10. Binary Search          : 10 problems


## **Step 3: Sandboxed Subprocess Test Judge (Phase 1 Deliverable)**
Enforces strict timeouts (2.0s) and memory caps (512MB) in isolated subprocesses.

In [5]:
class Verdict(str, Enum):
    AC = "AC"    # Accepted
    WA = "WA"    # Wrong Answer
    TLE = "TLE"  # Time Limit Exceeded
    MLE = "MLE"  # Memory Limit Exceeded
    RE = "RE"    # Runtime Error
    CE = "CE"    # Compilation / Syntax Error

@dataclass
class TestResult:
    test_index: int
    input_data: str
    expected_output: str
    actual_output: str
    verdict: Verdict
    runtime_ms: float
    error_message: Optional[str] = None

@dataclass
class ExecutionResult:
    verdict: Verdict
    tests_passed: int
    total_tests: int
    pass_rate: float
    max_runtime_ms: float
    test_results: List[TestResult] = field(default_factory=list)

class SandboxJudge:
    def __init__(self, default_timeout_s: float = 2.0, max_memory_mb: int = 512):
        self.default_timeout_s = default_timeout_s
        self.max_memory_mb = max_memory_mb

    @staticmethod
    def _normalize_output(text: str) -> List[str]:
        if not text:
            return []
        lines = text.strip().replace("\r\n", "\n").split("\n")
        return [line.rstrip() for line in lines if line.rstrip()]

    def _set_limits(self, memory_mb: int) -> None:
        try:
            mem_bytes = memory_mb * 1024 * 1024
            resource.setrlimit(resource.RLIMIT_AS, (mem_bytes, mem_bytes))
        except (ValueError, OSError):
            pass

    def evaluate_python_solution(self, code: str, test_cases: List[Dict[str, str]], timeout_s: Optional[float] = None) -> ExecutionResult:
        timeout = timeout_s or self.default_timeout_s
        if not test_cases:
            return ExecutionResult(Verdict.AC, 0, 0, 1.0, 0.0)

        try:
            compile(code, "<string>", "exec")
        except SyntaxError as e:
            return ExecutionResult(Verdict.CE, 0, len(test_cases), 0.0, 0.0)

        with tempfile.NamedTemporaryFile(mode="w", suffix=".py", delete=False) as f:
            f.write(code)
            script_path = f.name

        try:
            results = []
            max_ms = 0.0
            overall_verdict = Verdict.AC
            for idx, tc in enumerate(test_cases):
                inp = tc.get("input_data", "")
                expected = tc.get("output_data", "")
                start_t = time.perf_counter()
                try:
                    proc = subprocess.run(
                        [sys.executable, script_path],
                        input=inp,
                        text=True,
                        capture_output=True,
                        timeout=timeout,
                        preexec_fn=lambda: self._set_limits(self.max_memory_mb)
                    )
                    runtime_ms = (time.perf_counter() - start_t) * 1000.0
                    max_ms = max(max_ms, runtime_ms)
                    if proc.returncode != 0:
                        verdict = Verdict.RE
                        if overall_verdict == Verdict.AC: overall_verdict = Verdict.RE
                        results.append(TestResult(idx, inp, expected, proc.stdout, verdict, runtime_ms, proc.stderr[:200]))
                        continue
                    norm_exp = self._normalize_output(expected)
                    norm_act = self._normalize_output(proc.stdout)
                    verdict = Verdict.AC if norm_exp == norm_act else Verdict.WA
                    if verdict != Verdict.AC and overall_verdict == Verdict.AC:
                        overall_verdict = Verdict.WA
                    results.append(TestResult(idx, inp, expected, proc.stdout, verdict, runtime_ms))
                except subprocess.TimeoutExpired:
                    runtime_ms = timeout * 1000.0
                    max_ms = max(max_ms, runtime_ms)
                    if overall_verdict == Verdict.AC: overall_verdict = Verdict.TLE
                    results.append(TestResult(idx, inp, expected, "", Verdict.TLE, runtime_ms))
            passed = sum(1 for r in results if r.verdict == Verdict.AC)
            return ExecutionResult(overall_verdict, passed, len(test_cases), passed / len(test_cases), max_ms, results)
        finally:
            if os.path.exists(script_path):
                os.remove(script_path)

judge = SandboxJudge()
print("=== Testing Sandboxed Judge on Sample Solutions ===")
for item in dataset[:5]:
    res = judge.evaluate_python_solution(item.teacher_solution, item.sample_tests)
    print(f"Problem {item.problem_id} ({item.ground_truth_tag}): Verdict = {res.verdict.value}, Passed = {res.tests_passed}/{res.total_tests}, Max Runtime = {res.max_runtime_ms:.1f}ms")

=== Testing Sandboxed Judge on Sample Solutions ===
Problem TACO-DY-01 (dynamic programming): Verdict = WA, Passed = 1/3, Max Runtime = 405.5ms
Problem TACO-DY-02 (dynamic programming): Verdict = WA, Passed = 0/3, Max Runtime = 2000.0ms
Problem TACO-DY-03 (dynamic programming): Verdict = AC, Passed = 3/3, Max Runtime = 183.2ms
Problem TACO-DY-04 (dynamic programming): Verdict = RE, Passed = 0/2, Max Runtime = 192.6ms
Problem TACO-DY-05 (dynamic programming): Verdict = RE, Passed = 0/3, Max Runtime = 362.2ms


## **Step 4: Rationale-Consistency Semantic Filter (Phase 2 Deliverable)**
Purifies the teacher distillation dataset before model training.

In [6]:
PARADIGM_SIGNATURES = {
    "dynamic programming": ["dynamic programming", "dp", "memoization", "subproblem", "state transition", "optimal substructure", "knapsack"],
    "greedy": ["greedy", "locally optimal", "greedy choice", "sort and choose", "exchange argument", "greedy allocation"],
    "graphs": ["graph", "vertex", "vertices", "edge", "edges", "dfs", "bfs", "connected component", "shortest path", "traversal"],
    "math": ["math", "mathematical", "formula", "parity", "even", "odd", "combinatorics", "equation"],
    "data structures": ["data structure", "heap", "priority queue", "hash map", "hash table", "stack", "queue", "dsu", "frequency array"],
    "trees": ["tree", "root", "leaf", "ancestor", "subtree", "depth", "forest", "hierarchy"],
    "brute force": ["brute force", "simulate", "simulation", "iterate", "try all", "exhaustive", "scan each"],
    "strings": ["string", "character", "characters", "substring", "prefix", "suffix", "vowel", "consonant", "lexicographical"],
    "number theory": ["number theory", "prime", "primes", "divisor", "divisors", "gcd", "lcm", "modulo", "modular", "sieve"],
    "binary search": ["binary search", "search space", "monotonic", "bisect", "upper_bound", "lower_bound", "logarithmic"],
}

class FilterVerdict(str, Enum):
    RETAINED = "RETAINED"
    REJECTED_TAG_MISMATCH = "REJECTED_TAG_MISMATCH"
    REJECTED_INSUFFICIENT_REASONING = "REJECTED_INSUFFICIENT_REASONING"
    REJECTED_CONTRADICTORY_LOGIC = "REJECTED_CONTRADICTORY_LOGIC"

@dataclass
class FilterResult:
    problem_id: str
    ground_truth_tag: str
    verdict: FilterVerdict
    is_valid: bool
    consistency_score: float
    reasons: List[str]

class RationaleFilter:
    def __init__(self, min_length: int = 40, threshold: float = 0.5):
        self.min_length = min_length
        self.threshold = threshold

    def evaluate_item(self, item: DatasetItem) -> FilterResult:
        gt_tag = item.ground_truth_tag.lower().strip()
        stated_tag = item.teacher_stated_tag.lower().strip() if item.teacher_stated_tag else ""
        rationale = item.teacher_rationale.lower()

        if stated_tag and stated_tag != gt_tag:
            for other_tag in PARADIGM_SIGNATURES:
                if other_tag != gt_tag and other_tag in stated_tag:
                    return FilterResult(item.problem_id, gt_tag, FilterVerdict.REJECTED_TAG_MISMATCH, False, 0.0, [f"Contradictory tag '{stated_tag}'"])

        if len(rationale.split()) < 8 or len(rationale) < self.min_length:
            return FilterResult(item.problem_id, gt_tag, FilterVerdict.REJECTED_INSUFFICIENT_REASONING, False, 0.1, ["Insufficient length"])

        target_kws = PARADIGM_SIGNATURES.get(gt_tag, [gt_tag])
        matches = [kw for kw in target_kws if re.search(r'\b' + re.escape(kw) + r'\b', rationale)]

        score = min(1.0, 0.4 + 0.2 * len(matches)) if matches else (0.6 if gt_tag in rationale else 0.2)
        verdict = FilterVerdict.RETAINED if score >= self.threshold else FilterVerdict.REJECTED_INSUFFICIENT_REASONING
        return FilterResult(item.problem_id, gt_tag, verdict, verdict == FilterVerdict.RETAINED, score, matches)

    def filter_dataset(self, dataset: List[DatasetItem]):
        retained, discarded = [], []
        for item in dataset:
            res = self.evaluate_item(item)
            if res.is_valid:
                item.is_consistency_verified = True
                retained.append(item)
            else:
                discarded.append(item)
        return retained, discarded

filt = RationaleFilter()
retained, discarded = filt.filter_dataset(dataset)
print(f"Rationale-Consistency Filter: Retained = {len(retained)}/{len(dataset)} ({len(retained)/len(dataset)*100:.1f}%), Discarded = {len(discarded)}")

# Dataset Partitioning (70% Train, 10% Val, 20% Test)
random.seed(42)
tag_groups = defaultdict(list)
for item in retained:
    tag_groups[item.ground_truth_tag].append(item)

train_split, val_split, test_split = [], [], []
for tag, items in tag_groups.items():
    shuffled = list(items)
    random.shuffle(shuffled)
    n = len(shuffled)
    n_train = max(1, int(n * 0.7))
    n_val = max(1, int(n * 0.1))
    train_split.extend(shuffled[:n_train])
    val_split.extend(shuffled[n_train:n_train + n_val])
    test_split.extend(shuffled[n_train + n_val:])

print(f"Distillation Dataset Partition: Train = {len(train_split)}, Val = {len(val_split)}, Test = {len(test_split)}")

class DistillationProblemDataset(Dataset):
    def __init__(self, items: List[DatasetItem], tokenizer, max_length: int = 256):
        self.items = items
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        item = self.items[idx]
        text = f"Problem: {item.title}\n{item.statement}\nInput: {item.input_spec}"
        encoding = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(TAG_TO_IDX[item.ground_truth_tag], dtype=torch.long)
        }

Rationale-Consistency Filter: Retained = 81/100 (81.0%), Discarded = 19
Distillation Dataset Partition: Train = 55, Val = 9, Test = 17


## **Step 5: Hugging Face PEFT + BitsAndBytes QLoRA Fine-Tuning**
Standard fine-tuning pipeline utilizing Hugging Face `peft` and `bitsandbytes` 4-bit NormalFloat (NF4) quantization on `Qwen/Qwen2.5-0.5B`.

In [7]:
STUDENT_MODEL_ID = "Qwen/Qwen2.5-0.5B"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"\n[PEFT Pipeline] Initializing HF PEFT + BitsAndBytes on {STUDENT_MODEL_ID}...")
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
tokenizer_peft = AutoTokenizer.from_pretrained(STUDENT_MODEL_ID, trust_remote_code=True)
if tokenizer_peft.pad_token is None:
    tokenizer_peft.pad_token = tokenizer_peft.eos_token

if torch.cuda.is_available():
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
        bnb_4bit_use_double_quant=True
    )
    base_model_peft = AutoModelForSequenceClassification.from_pretrained(
        STUDENT_MODEL_ID,
        num_labels=len(TARGET_TAG_TAXONOMY),
        quantization_config=bnb_config,
        device_map="auto"
    )
    base_model_peft = prepare_model_for_kbit_training(base_model_peft)
else:
    base_model_peft = AutoModelForSequenceClassification.from_pretrained(
        STUDENT_MODEL_ID,
        num_labels=len(TARGET_TAG_TAXONOMY)
    ).to(device)

base_model_peft.config.pad_token_id = tokenizer_peft.pad_token_id

lora_config_peft = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none"
)

model_peft = get_peft_model(base_model_peft, lora_config_peft)
model_peft.print_trainable_parameters()

# Train PEFT Model
train_dataset_peft = DistillationProblemDataset(train_split, tokenizer_peft)
train_loader_peft = DataLoader(train_dataset_peft, batch_size=4, shuffle=True)
optimizer_peft = torch.optim.AdamW(model_peft.parameters(), lr=2e-4, weight_decay=0.01)
num_epochs = 4

peft_train_losses = []
start_t_peft = time.perf_counter()
model_peft.train()

for epoch in range(1, num_epochs + 1):
    epoch_loss = 0.0
    for batch in train_loader_peft:
        optimizer_peft.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model_peft(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer_peft.step()
        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_loader_peft)
    peft_train_losses.append(avg_loss)
    print(f"  [HF PEFT] Epoch {epoch}/{num_epochs} Loss: {avg_loss:.4f}")

peft_total_time = time.perf_counter() - start_t_peft
peft_peak_vram_mb = get_gpu_memory_mb()
print(f"✓ [HF PEFT] Training Finished in {peft_total_time:.2f}s | Peak VRAM: {peft_peak_vram_mb:.1f} MB")


[PEFT Pipeline] Initializing HF PEFT + BitsAndBytes on Qwen/Qwen2.5-0.5B...


config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-0.5B
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


trainable params: 2,171,648 || all params: 496,213,376 || trainable%: 0.4376


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


  [HF PEFT] Epoch 1/4 Loss: 7.2084
  [HF PEFT] Epoch 2/4 Loss: 1.9741
  [HF PEFT] Epoch 3/4 Loss: 0.8432
  [HF PEFT] Epoch 4/4 Loss: 0.3300
✓ [HF PEFT] Training Finished in 65.01s | Peak VRAM: 992.6 MB


## **Step 6: Empirical Evaluation & Performance Metrics**
Evaluates the fine-tuned model on the unseen test partition and visualizes classification accuracy and convergence:

In [8]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def evaluate_model(model, tokenizer, test_items):
    model.eval()
    test_dataset = DistillationProblemDataset(test_items, tokenizer)
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)
    y_true = [item.ground_truth_tag for item in test_items]
    y_pred = []
    y_probs = []

    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]
            pred_idx = int(np.argmax(probs))
            y_pred.append(IDX_TO_TAG[pred_idx])
            y_probs.append(probs)

    acc = accuracy_score(y_true, y_pred)
    top_3_correct = 0
    for i, true_tag in enumerate(y_true):
        top_3_indices = np.argsort(y_probs[i])[-3:]
        top_3_tags = [IDX_TO_TAG[idx] for idx in top_3_indices]
        if true_tag in top_3_tags:
            top_3_correct += 1
    top_3_acc = top_3_correct / len(y_true)
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, labels=TARGET_TAG_TAXONOMY, average="macro", zero_division=0)
    return acc, top_3_acc, f1, y_pred, y_probs

peft_acc, peft_top3, peft_f1, _, _ = evaluate_model(model_peft, tokenizer_peft, test_split)

print("\n=========================================================================")
print("  📊 EVALUATION BENCHMARK: HUGGING FACE PEFT + BITSANDBYTES             ")
print("=========================================================================")
print(f"{'Metric':<35} | {'Value':<20}")
print("-" * 58)
print(f"{'Base Student LLM':<35} | {STUDENT_MODEL_ID:<20}")
print(f"{'Quantization Precision':<35} | {'4-bit NF4':<20}")
print(f"{'Trainable Parameters':<35} | {'2.17M (0.44%)':<20}")
print(f"{'Total Training Time':<35} | {f'{peft_total_time:.2f}s':<20}")
print(f"{'Peak VRAM Memory':<35} | {f'{peft_peak_vram_mb:.1f} MB':<20}")
print(f"{'Top-1 Classification Accuracy':<35} | {f'{peft_acc*100:.2f}%':<20}")
print(f"{'Top-3 Classification Accuracy':<35} | {f'{peft_top3*100:.2f}%':<20}")
print(f"{'Macro F1 Score':<35} | {f'{peft_f1:.4f}':<20}")
print("=" * 58)

# Plot Convergence Curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
epochs = list(range(1, num_epochs + 1))
axes[0].plot(epochs, peft_train_losses, marker="o", linewidth=2.5, color="#4A90E2", label="PEFT Loss")
axes[0].set_xlabel("Epoch", fontweight="bold")
axes[0].set_ylabel("Cross-Entropy Loss", fontweight="bold")
axes[0].set_title("Training Loss Trajectory", pad=10, fontweight="bold")
axes[0].grid(True, linestyle="--", alpha=0.5)
axes[0].legend()

acc_metrics = ["Top-1 Acc", "Top-3 Acc", "Macro F1"]
acc_vals = [peft_acc * 100, peft_top3 * 100, peft_f1 * 100]
axes[1].bar(acc_metrics, acc_vals, color=["#50E3C2", "#4A90E2", "#E94E77"], width=0.5)
axes[1].set_ylabel("Percentage (%)", fontweight="bold")
axes[1].set_title("Test Set Accuracy Metrics", pad=10, fontweight="bold")
for i, v in enumerate(acc_vals):
    axes[1].text(i, v + 2, f"{v:.1f}%", ha="center", fontweight="bold")
axes[1].grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()


  📊 EVALUATION BENCHMARK: HUGGING FACE PEFT + BITSANDBYTES             
Metric                              | Value               
----------------------------------------------------------
Base Student LLM                    | Qwen/Qwen2.5-0.5B   
Quantization Precision              | 4-bit NF4           
Trainable Parameters                | 2.17M (0.44%)       
Total Training Time                 | 65.01s              
Peak VRAM Memory                    | 992.6 MB            
Top-1 Classification Accuracy       | 17.65%              
Top-3 Classification Accuracy       | 41.18%              
Macro F1 Score                      | 0.0952              


## **Step 7: Interactive Live Problem Solver**
Run live predictions with the fine-tuned model on custom competitive programming problem statements:

In [9]:
def predict_problem(problem_statement: str, top_k: int = 3):
    model_peft.eval()
    enc = tokenizer_peft(problem_statement, truncation=True, max_length=256, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = model_peft(**enc).logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]
    sorted_idx = np.argsort(probs)[::-1]
    return [(IDX_TO_TAG[idx], float(probs[idx])) for idx in sorted_idx[:top_k]]

demo_problem = """
Given a weighted undirected graph with n vertices and m edges. Find the shortest path
between vertex 1 and vertex n using Dijkstra's algorithm with a min-priority queue.
"""

preds = predict_problem(demo_problem)
print("=== LIVE INFERENCE PREDICTION ===")
print(f"Problem Statement:\n{demo_problem.strip()}\n")
print(f"Predicted Algorithmic Paradigm: {preds[0][0].upper()} (Confidence: {preds[0][1]*100:.1f}%)\n")
print("Top-3 Confidence Distribution:")
for r, (tag, prob) in enumerate(preds, 1):
    print(f"  {r}. {tag.title():<22} [{('█'*int(prob*25)):<25}] {prob*100:.1f}%")

=== LIVE INFERENCE PREDICTION ===
Problem Statement:
Given a weighted undirected graph with n vertices and m edges. Find the shortest path
between vertex 1 and vertex n using Dijkstra's algorithm with a min-priority queue.

Predicted Algorithmic Paradigm: DYNAMIC PROGRAMMING (Confidence: 98.3%)

Top-3 Confidence Distribution:
  1. Dynamic Programming    [████████████████████████ ] 98.3%
  2. Graphs                 [                         ] 1.1%
  3. Trees                  [                         ] 0.5%


### **✓ Midsemester Evaluation Verification (40% Scope Complete)**
- **Phase 1 (20%)**: 100 TACO benchmark problems, Google Gemini teacher traces, and Subprocess Sandbox Judge.
- **Phase 2 (20%)**: Rationale-Consistency Filter, **Hugging Face PEFT + BitsAndBytes QLoRA Distillation** on `Qwen2.5-0.5B`, Evaluation Dashboard, and Live Inference.
- **Evaluation Target**: Ready for demonstration and evaluation.